# ⚡ Project 04: EV "Charging Deserts" & Spatial Infrastructure Analytics
### Geospatial Machine Learning, Urban Accessibility & Clean Energy Planning

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟢 Beginner  
**Domain:** Clean Energy & Urban Tech  

---
### Notebook Outline:
1. **Environment Setup & Geospatial Imports**
2. **Data Ingestion & Regional Coordinates Exploration**
3. **Supply-Demand Deficit Index Formulation**
4. **Spatial Density Visualizations & Interactive Distribution Plots**
5. **K-Means Spatial Regional Partitioning**
6. **DBSCAN Density-Based Outlier Detection (Locating Charging Deserts)**
7. **Infrastructure Prioritization & Capital Allocation Matrix**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Energy geospatial workspace ready.")

In [ ]:
# Data Ingestion
df = pd.read_csv("data/ev_charging_infrastructure.csv")
print(f"Urban Zones Analyzed: {len(df)}")
display(df.head(4))

In [ ]:
# Supply-Demand Deficit Index Formulation
# CDI represents EV demand pressure per available charging plug
df['charging_deficit_index'] = (
    (df['registered_ev_count'] + 0.05 * df['daily_traffic_flow']) /
    (df['existing_chargers'] + 1.0)
)

print("Top 5 Acute Charging Desert Zones:")
display(df.sort_values('charging_deficit_index', ascending=False)[
    ['zone_id', 'latitude', 'longitude', 'registered_ev_count', 'existing_chargers', 'charging_deficit_index']
].head(5))

In [ ]:
# Spatial Plotting: Existing Chargers vs. Deficit Index
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(df['longitude'], df['latitude'], c=df['existing_chargers'], cmap='viridis', s=35, alpha=0.8)
axes[0].set_title("Existing EV Charger Plugs", fontweight='bold')
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
plt.colorbar(sc1, ax=axes[0], label="Chargers Count")

sc2 = axes[1].scatter(df['longitude'], df['latitude'], c=df['charging_deficit_index'], cmap='Reds', s=35, alpha=0.8)
axes[1].set_title("Charging Deficit Index (Red = Severe Desert)", fontweight='bold')
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(sc2, ax=axes[1], label="Deficit Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# Spatial Machine Learning: DBSCAN for Dense Hubs and Outlier Deserts
geo_coords = df[['latitude', 'longitude']].values
geo_scaled = StandardScaler().fit_transform(geo_coords)

# Fit DBSCAN
dbscan = DBSCAN(eps=0.25, min_samples=15)
df['dbscan_cluster'] = dbscan.fit_predict(geo_scaled)

print(f"Clusters Detected: {len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'] else 0)}")
print(f"Isolated Zones (Deserts / Outliers): {(df['dbscan_cluster'] == -1).sum()}")

# Prioritization Score: High Deficit + High Grid Capacity
df['deployment_priority_score'] = (
    df['charging_deficit_index'] * 0.6 + 
    df['power_grid_capacity_mw'] * 15.0
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x='longitude', y='latitude', hue='dbscan_cluster',
    palette='tab10', style=(df['dbscan_cluster'] == -1), s=50
)
plt.title("DBSCAN Spatial Cluster Partitioning & Isolated Outliers", fontweight='bold')
plt.show()

In [ ]:
# Executive Decision Matrix: Top 10 Recommended Sites for Next-Gen DC Fast Charging
top_deployments = df.sort_values('deployment_priority_score', ascending=False).head(10)
print("=== Top 10 Priority Locations for Capital Deployment ===")
display(top_deployments[['zone_id', 'latitude', 'longitude', 'registered_ev_count', 
                         'existing_chargers', 'power_grid_capacity_mw', 'deployment_priority_score']].round(2))